# Генераторы. Подтверждающий анализ

## Лекция

In [1]:
import pandas as pd
import numpy as np
import random

# seed for reproducibility
np.random.seed(42)
random.seed(42)
n = 1000

# ----- ГЕНЕРАЦИЯ ПРИЗНАКОВ -----

# patient_id
patient_id = list(range(1, n+1))

# age (0-100)
age = np.random.randint(18, 85, n)

# doctor_id (20 doctors)
doctor_id = np.random.randint(1, 21, n)

# specialization
spec_map = {
    **{i: 'терапевт' for i in range(1, 6)},
    **{i: 'кардиолог' for i in range(6, 10)},
    **{i: 'невролог' for i in range(10, 14)},
    **{i: 'хирург' for i in range(14, 21)}
}
specialization = [spec_map[d] for d in doctor_id]

# primary_symptoms_score (1-10)
primary_symptoms_score = np.random.randint(1, 11, n)

# prescribed_treatment
treatment_pool = ['лекарства', 'процедуры', 'наблюдение', 'направление к другому специалисту']
prescribed_treatment = [random.choice(treatment_pool) for _ in range(n)]

# visited_again_30days (base 35% chance, but boosted for heavy patients)
def visit_logic(spec, symp):
    if spec in ['хирург', 'невролог'] and symp >= 7:
        return 'Да' if np.random.random() < 0.75 else 'Нет'
    elif symp >= 8:
        return 'Да' if np.random.random() < 0.6 else 'Нет'
    else:
        return 'Да' if np.random.random() < 0.25 else 'Нет'

visited_again_30days = [visit_logic(spec, symp) for spec, symp in zip(specialization, primary_symptoms_score)]

# invoice_amount with outliers for doctor_id 5
def invoice_logic(doctor_id, symp):
    base = np.random.normal(1500, 300)
    if doctor_id == 5:
        # doctor 5: overcharges
        overcharge_factor = np.random.choice([2.5, 3.0, 1.2, 1.0], p=[0.3, 0.2, 0.3, 0.2])
        base = base * overcharge_factor
    elif symp >= 8:
        base += np.random.randint(500, 1500)
    return max(round(base, 2), 200)

invoice_amount = [invoice_logic(d, symp) for d, symp in zip(doctor_id, primary_symptoms_score)]

# ---- ПРОПУСКИ ----

# A*3: satisfaction_score (MCAR, 12%)
satisfaction_score = np.random.randint(0, 11, n).astype(float)
missing_sat = np.random.choice([True, False], n, p=[0.12, 0.88])
satisfaction_score[missing_sat] = np.nan

# B*1: gender (missing more for surgeon, rest random)
gender_list = []
for spec in specialization:
    if spec == 'хирург' and np.random.random() < 0.30:
        gender_list.append(np.nan)
    elif np.random.random() < 0.10:
        gender_list.append(np.nan)
    else:
        gender_list.append(np.random.choice(['М', 'Ж']))

# Замена пропусков по правилу B*1 (делаем это уже в сгенерированном наборе, чтобы показать студентам)
# для хирургов мода 'М', остальные -> 'Не указан'
gender_filled = []
for spec, g in zip(specialization, gender_list):
    if pd.isna(g):
        if spec == 'хирург':
            gender_filled.append('М')
        else:
            gender_filled.append('Не указан')
    else:
        gender_filled.append(g)

# Далее — замена пропусков в satisfaction_score на общую медиану (A*3)
median_sat = np.nanmedian(satisfaction_score)
satisfaction_filled = satisfaction_score.copy()
satisfaction_filled[pd.isna(satisfaction_filled)] = median_sat
satisfaction_filled = satisfaction_filled.round(1)

# ----- ИТОГОВЫЙ ДАТАФРЕЙМ -----
df = pd.DataFrame({
    'patient_id': patient_id,
    'age': age,
    'gender_raw': gender_list,          # with original NaN
    'gender': gender_filled,            # after filling
    'doctor_id': doctor_id,
    'specialization': specialization,
    'primary_symptoms_score': primary_symptoms_score,
    'prescribed_treatment': prescribed_treatment,
    'visited_again_30days': visited_again_30days,
    'satisfaction_score_raw': satisfaction_score,   # with original NaN
    'satisfaction_score': satisfaction_filled,      # after filling
    'invoice_amount': invoice_amount
})

# Сохраняем
df.to_csv('clinic_data_otter.csv', index=False)

# Выводим первые 10 строк для проверки
print(df.head(10))
print(f"\nПропусков в gender_raw: {df['gender_raw'].isna().sum()}")
print(f"Пропусков в satisfaction_score_raw: {df['satisfaction_score_raw'].isna().sum()}")


   patient_id  age gender_raw     gender  doctor_id specialization  \
0           1   69        NaN  Не указан         11       невролог   
1           2   32          М          М          6      кардиолог   
2           3   78          Ж          Ж         18         хирург   
3           4   38          Ж          Ж         10       невролог   
4           5   41          М          М          4       терапевт   
5           6   20          М          М          9      кардиолог   
6           7   39        NaN  Не указан          6      кардиолог   
7           8   70          М          М          2       терапевт   
8           9   19        NaN          М         17         хирург   
9          10   47          Ж          Ж          5       терапевт   

   primary_symptoms_score               prescribed_treatment  \
0                       5                          лекарства   
1                       9                          лекарства   
2                       3            

In [1]:
import pandas as pd
import numpy as np
import random

# seed for reproducibility
np.random.seed(42)
random.seed(42)
n = 800

# ----- ГЕНЕРАЦИЯ ПРИЗНАКОВ -----

# house_id
house_id = list(range(1, n+1))

# district
districts = ['Центральный', 'Заводской', 'Ленинский', 'Приречный']
district = np.random.choice(districts, n, p=[0.25, 0.25, 0.25, 0.25])

# house_age (5-100)
house_age = np.random.randint(5, 101, n)

# floors (1-25, but older houses may be taller for some)
floors = [np.random.randint(1, 6) if h < 30 else np.random.randint(5, 20) for h in house_age]

# management_company_id (20 УК)
management_company_id = np.random.randint(1, 21, n)

# contract_amount: base, but higher for old+tall houses handled by id 7 and 12
def contract_logic(mc_id, age, fl):
    base = np.random.normal(8_000_000, 1_500_000)
    if mc_id in [7, 12] and age > 60 and fl >= 9:
        base = base * np.random.uniform(1.8, 2.5)
    elif age > 60 and fl >= 9:
        base = base * np.random.uniform(1.3, 1.7)
    return max(round(base, 2), 1_000_000)

contract_amount = [contract_logic(mc, age, fl) for mc, age, fl in zip(management_company_id, house_age, floors)]

# planned_duration_days
def planned_logic(mc_id, age, fl):
    base = np.random.randint(60, 180)
    if mc_id == 15:
        # systematically underestimates
        base = np.random.randint(30, 80)
    if age > 60:
        base += np.random.randint(20, 60)
    if fl >= 9:
        base += np.random.randint(10, 40)
    return base

planned_duration_days = [planned_logic(mc, age, fl) for mc, age, fl in zip(management_company_id, house_age, floors)]

# actual_duration_days (always >= planned, especially for id 15)
actual_duration_days = []
for planned, mc in zip(planned_duration_days, management_company_id):
    if mc == 15:
        actual = planned + np.random.randint(60, 200)
    else:
        actual = planned + np.random.randint(-20, 50)
        actual = max(actual, planned - 10)
    actual_duration_days.append(actual)

# citizen_complaints (poisson, but hidden for mc 3 and 8)
citizen_complaints_raw = []
for mc in management_company_id:
    if mc in [3, 8]:
        complaints = np.random.poisson(lam=5)
    else:
        complaints = np.random.poisson(lam=2)
    citizen_complaints_raw.append(complaints)

# inspection_score (0-100 quality, some missing)
inspection_score_raw = []
for planned, actual, cct in zip(planned_duration_days, actual_duration_days, citizen_complaints_raw):
    base = np.random.normal(70, 15)
    # delays and complaints reduce score
    delay = max(0, actual - planned)
    score = base - delay * 0.2 - cct * 3
    score = max(0, min(100, score))
    inspection_score_raw.append(round(score, 1))

# budget_overrun_pct
budget_overrun_pct = []
for mc, planned, actual, insp_score in zip(management_company_id, planned_duration_days, actual_duration_days, inspection_score_raw):
    if mc == 15 and insp_score < 50:
        overrun = np.random.uniform(120, 180)
    else:
        overrun = (actual - planned) / planned * 100 + np.random.normal(0, 10)
        overrun = max(-30, min(overrun, 100))
    budget_overrun_pct.append(round(overrun, 1))

# ---- ПРОПУСКИ ----

# A*3: inspection_score (MCAR, 8%)
inspection_score_nan = inspection_score_raw.copy()
missing_insp = np.random.choice([True, False], n, p=[0.08, 0.92])
for i in range(n):
    if missing_insp[i]:
        inspection_score_nan[i] = np.nan

# B*1: citizen_complaints (missing more for mc 3 and 8)
citizen_complaints_nan = citizen_complaints_raw.copy()
for i in range(n):
    mc = management_company_id[i]
    if mc in [3, 8] and np.random.random() < 0.40:
        citizen_complaints_nan[i] = np.nan
    elif np.random.random() < 0.03:
        citizen_complaints_nan[i] = np.nan

# Замена пропусков по правилу B*1:
# для mc 3 и 8 — медиана по mc-группе, остальные — 0
citizen_complaints_filled = []
for i in range(n):
    val = citizen_complaints_nan[i]
    if pd.isna(val):
        mc = management_company_id[i]
        if mc in [3, 8]:
            mc_group_vals = [citizen_complaints_nan[j] for j in range(n) if management_company_id[j] == mc and not pd.isna(citizen_complaints_nan[j])]
            if mc_group_vals:
                citizen_complaints_filled.append(round(np.median(mc_group_vals)))
            else:
                citizen_complaints_filled.append(0)
        else:
            citizen_complaints_filled.append(0)
    else:
        citizen_complaints_filled.append(val)

# Замена пропусков в inspection_score на общую медиану (A*3)
median_insp = np.nanmedian(inspection_score_nan)
inspection_score_filled = [round(median_insp, 1) if pd.isna(v) else v for v in inspection_score_nan]

# ----- ИТОГОВЫЙ ДАТАФРЕЙМ -----
df = pd.DataFrame({
    'house_id': house_id,
    'district': district,
    'house_age': house_age,
    'floors': floors,
    'management_company_id': management_company_id,
    'contract_amount': contract_amount,
    'planned_duration_days': planned_duration_days,
    'actual_duration_days': actual_duration_days,
    'citizen_complaints_raw': citizen_complaints_nan,      # with NaN
    'citizen_complaints': citizen_complaints_filled,       # after filling B*1
    'inspection_score_raw': inspection_score_nan,          # with NaN
    'inspection_score': inspection_score_filled,           # after filling A*3
    'budget_overrun_pct': budget_overrun_pct
})

# Сохраняем
df.to_csv('severnorechensk_housing_audit.csv', index=False)

# Выводим первые 10 строк для проверки
print(df.head(10))
print(f"\nПропусков в citizen_complaints_raw: {df['citizen_complaints_raw'].isna().sum()}")
print(f"Пропусков в inspection_score_raw: {df['inspection_score_raw'].isna().sum()}")


   house_id     district  house_age  floors  management_company_id  \
0         1    Заводской         76       7                     11   
1         2    Приречный         96      16                      1   
2         3    Ленинский         35       6                     13   
3         4    Ленинский         13       2                     19   
4         5  Центральный         55       7                      3   
5         6  Центральный         33      10                     17   
6         7  Центральный         82       9                      5   
7         8    Приречный         44      12                     20   
8         9    Ленинский         45      13                      8   
9        10    Ленинский         90      15                     14   

   contract_amount  planned_duration_days  actual_duration_days  \
0      11049940.65                    103                   141   
1      11330595.41                    209                   199   
2       8256247.85          

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 800 entries, 0 to 799
Data columns (total 13 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   house_id                800 non-null    int64  
 1   district                800 non-null    object 
 2   house_age               800 non-null    int32  
 3   floors                  800 non-null    int64  
 4   management_company_id   800 non-null    int32  
 5   contract_amount         800 non-null    float64
 6   planned_duration_days   800 non-null    int64  
 7   actual_duration_days    800 non-null    int64  
 8   citizen_complaints_raw  736 non-null    float64
 9   citizen_complaints      800 non-null    int64  
 10  inspection_score_raw    741 non-null    float64
 11  inspection_score        800 non-null    float64
 12  budget_overrun_pct      800 non-null    float64
dtypes: float64(5), int32(2), int64(5), object(1)
memory usage: 75.1+ KB


In [9]:
import pandas as pd
import numpy as np
import random

# seed for reproducibility
np.random.seed(42)
random.seed(42)
n = 1000

# ----- ГЕНЕРАЦИЯ ПРИЗНАКОВ -----

# client_id
client_id = list(range(1, n+1))

# age (18-80)
age = np.random.randint(18, 81, n)

# manager_id (15 managers)
manager_id = np.random.randint(1, 16, n)

# destination (weighted by manager)
dest_pools = {
    'Турция': [1, 2, 3],
    'Египет': [4, 5, 6],
    'Таиланд': [7, 8, 9],
    'ОАЭ': [10, 11],
    'Мальдивы': [12, 13],
    'Европа': [14, 15]
}
# Build reverse mapping: manager -> primary destination
manager_dest = {}
for dest, mgrs in dest_pools.items():
    for m in mgrs:
        manager_dest[m] = dest

destination = []
for mid in manager_id:
    # 70% primary destination, 30% random
    if np.random.random() < 0.70:
        destination.append(manager_dest[mid])
    else:
        destination.append(np.random.choice(list(dest_pools.keys())))

# tour_duration_days
def duration_logic(dest):
    if dest in ['Мальдивы', 'ОАЭ']:
        return np.random.randint(7, 15)
    elif dest == 'Европа':
        return np.random.randint(5, 21)
    else:
        return np.random.randint(3, 14)

tour_duration_days = [duration_logic(d) for d in destination]

# extras_count (base: 0-10, boosted for premium destinations + certain managers)
extras_count = []
for mid, dest in zip(manager_id, destination):
    if dest in ['Мальдивы', 'ОАЭ']:
        extras = np.random.randint(3, 11)
    elif mid == 9:
        # manager 9 pushes extras
        extras = np.random.choice([3, 4, 5, 6, 7, 8], p=[0.1, 0.1, 0.2, 0.2, 0.2, 0.2])
    else:
        extras = np.random.randint(0, 7)
    extras_count.append(extras)

# tour_cost_rub (base + duration + extras + outliers for manager 9)
def cost_logic(mid, dest, dur, ext):
    base_cost = {
        'Турция': 40000,
        'Египет': 45000,
        'Таиланд': 60000,
        'ОАЭ': 80000,
        'Мальдивы': 120000,
        'Европа': 90000
    }
    base = base_cost[dest]
    cost = base + dur * 3000 + ext * 8000
    if mid == 9:
        # manager 9 inflates cost
        cost = cost * np.random.choice([1.5, 1.8, 2.0, 2.5], p=[0.3, 0.3, 0.2, 0.2])
    cost += np.random.normal(0, 10000)
    return max(round(cost, 2), 20000)

tour_cost_rub = [cost_logic(mid, dest, dur, ext) for mid, dest, dur, ext in zip(manager_id, destination, tour_duration_days, extras_count)]

# post_tour_rating (1-5, boosted for premium destinations)
def rating_logic(dest, ext):
    if dest in ['Мальдивы', 'ОАЭ']:
        base = np.random.choice([4, 5], p=[0.3, 0.7])
    elif ext >= 5:
        base = np.random.choice([3, 4, 5], p=[0.2, 0.4, 0.4])
    else:
        base = np.random.choice([1, 2, 3, 4, 5], p=[0.05, 0.1, 0.25, 0.35, 0.25])
    return base

post_tour_rating = [rating_logic(dest, ext) for dest, ext in zip(destination, extras_count)]

# refund_requested (higher for manager 9)
refund_requested = []
for mid, dest in zip(manager_id, destination):
    if mid == 9:
        refund_requested.append('Да' if np.random.random() < 0.35 else 'Нет')
    else:
        refund_requested.append('Да' if np.random.random() < 0.08 else 'Нет')

# ---- ПРОПУСКИ ----

# A*3: post_tour_rating (MCAR, 10%)
post_tour_rating_raw = post_tour_rating.copy()
missing_rating = np.random.choice([True, False], n, p=[0.10, 0.90])
for i in range(n):
    if missing_rating[i]:
        post_tour_rating_raw[i] = np.nan

# B*1: gender (missing more for ОАЭ, random otherwise)
gender_list = []
for dest in destination:
    if dest == 'ОАЭ' and np.random.random() < 0.35:
        gender_list.append(np.nan)
    elif np.random.random() < 0.07:
        gender_list.append(np.nan)
    else:
        gender_list.append(np.random.choice(['М', 'Ж']))

# Замена пропусков по правилу B*1:
# для ОАЭ мода 'М', остальные -> 'Не указан'
gender_filled = []
for dest, g in zip(destination, gender_list):
    if pd.isna(g):
        if dest == 'ОАЭ':
            gender_filled.append('М')
        else:
            gender_filled.append('Не указан')
    else:
        gender_filled.append(g)

# Замена пропусков в post_tour_rating на общую медиану (A*3)
median_rating = np.nanmedian(post_tour_rating_raw)
post_tour_rating_filled = [median_rating if pd.isna(v) else v for v in post_tour_rating_raw]
post_tour_rating_filled = [round(v, 1) for v in post_tour_rating_filled]

# ----- ИТОГОВЫЙ ДАТАФРЕЙМ -----
df = pd.DataFrame({
    'client_id': client_id,
    'age': age,
    'gender_raw': gender_list,                      # with original NaN
    'gender': gender_filled,                        # after filling B*1
    'manager_id': manager_id,
    'destination': destination,
    'tour_duration_days': tour_duration_days,
    'tour_cost_rub': tour_cost_rub,
    'extras_count': extras_count,
    'post_tour_rating_raw': post_tour_rating_raw,   # with original NaN
    'post_tour_rating': post_tour_rating_filled,    # after filling A*3
    'refund_requested': refund_requested
})

# Сохраняем
df.to_csv('suntravel_data.csv', index=False)

# Выводим первые 10 строк для проверки
print(df.head(10))
print(f"\nПропусков в gender_raw: {df['gender_raw'].isna().sum()}")
print(f"Пропусков в post_tour_rating_raw: {df['post_tour_rating_raw'].isna().sum()}")


   client_id  age gender_raw     gender  manager_id destination  \
0          1   56          М          М          15      Европа   
1          2   69        NaN  Не указан           9     Таиланд   
2          3   46          М          М           2      Турция   
3          4   32        NaN          М          10         ОАЭ   
4          5   60          Ж          Ж           8     Таиланд   
5          6   25          Ж          Ж          14      Европа   
6          7   78        NaN          М           2         ОАЭ   
7          8   38        NaN          М          11         ОАЭ   
8          9   56          М          М          13         ОАЭ   
9         10   75          Ж          Ж           5      Турция   

   tour_duration_days  tour_cost_rub  extras_count  post_tour_rating_raw  \
0                  16      172337.47             4                   5.0   
1                   5      196280.25             5                   5.0   
2                  13       99855.

In [10]:
# df.to_csv('data/travel_raw.csv', index=False)

In [11]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 12 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   client_id             1000 non-null   int64  
 1   age                   1000 non-null   int32  
 2   gender_raw            869 non-null    object 
 3   gender                1000 non-null   object 
 4   manager_id            1000 non-null   int32  
 5   destination           1000 non-null   object 
 6   tour_duration_days    1000 non-null   int64  
 7   tour_cost_rub         1000 non-null   float64
 8   extras_count          1000 non-null   int64  
 9   post_tour_rating_raw  900 non-null    float64
 10  post_tour_rating      1000 non-null   float64
 11  refund_requested      1000 non-null   object 
dtypes: float64(3), int32(2), int64(3), object(4)
memory usage: 86.1+ KB


In [12]:
df = df.drop(columns = ['gender', 'post_tour_rating'])
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 10 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   client_id             1000 non-null   int64  
 1   age                   1000 non-null   int32  
 2   gender_raw            869 non-null    object 
 3   manager_id            1000 non-null   int32  
 4   destination           1000 non-null   object 
 5   tour_duration_days    1000 non-null   int64  
 6   tour_cost_rub         1000 non-null   float64
 7   extras_count          1000 non-null   int64  
 8   post_tour_rating_raw  900 non-null    float64
 9   refund_requested      1000 non-null   object 
dtypes: float64(2), int32(2), int64(3), object(3)
memory usage: 70.4+ KB


In [13]:
df = df.rename(columns={'gender_raw': 'gender', 'post_tour_rating_raw': 'post_tour_rating'})
df

,client_id,age,gender,manager_id,destination,tour_duration_days,tour_cost_rub,extras_count,post_tour_rating,refund_requested
0,1,56,М,15,Европа,16,172337.47,4,5.0,Нет
1,2,69,NaN,9,Таиланд,5,196280.25,5,5.0,Да
2,3,46,М,2,Турция,13,99855.94,3,4.0,Нет
3,4,32,NaN,10,ОАЭ,14,187582.88,7,5.0,Да
4,5,60,Ж,8,Таиланд,4,101899.55,2,5.0,Нет
...,...,...,...,...,...,...,...,...,...,...
995,996,78,М,11,Египет,12,117835.95,6,5.0,Нет
996,997,23,М,10,ОАЭ,7,138062.62,5,4.0,Нет
997,998,35,М,7,Таиланд,3,72117.94,2,1.0,Нет
998,999,68,М,1,Мальдивы,12,179012.33,4,5.0,Нет


In [14]:
df.to_csv('data/travel.csv', index=False)